In [104]:
import pandas as pd
import datetime

# --- Date range: previous calendar month ---
today = datetime.date.today()
first_of_this_month = today.replace(day=1)
last_month_end = first_of_this_month - datetime.timedelta(days=1)
last_month_start = last_month_end.replace(day=1)
print(f"Reporting period: {last_month_start} → {last_month_end}")

# Canned HMIS report GNRL-210 Assesment Details Report, from EBH program - https://sac.clarityhs.com/report
ebh_path = r"isp_data\ebh_Individualized Support Plan (ISP)(676).xlsx"
grove = pd.read_excel(ebh_path, sheet_name="Individualized Support Plan (IS")
grove_es = grove[grove["Program Name"] == "CITY-FSC: Emergency Bridge Housing at The Grove - ES"]
grove_sso = grove[grove["Program Name"] == "CITY-FSC: Emergency Bridge Housing at The Grove - RRH"]

# Canned HMIS report GNRL-210 Assesment Details Report, from FSC program - https://sac.clarityhs.com/report
rrc_path = r"isp_data\dhsh-Individualized Support Plan (ISP)(676).xlsx"
rrc = pd.read_excel(rrc_path, sheet_name="Individualized Support Plan (IS")
roseville = rrc[rrc["Program Name"] == "FSC: Roseville Road - ES"]

# Custom HMIS report containing all service recieving clients - https://sac.clarityhs.com/report/embed/114988/1
sup_path = r"isp_data\monthly_supplament_april.csv"
sup = pd.read_csv(sup_path)

# Custom HMIS report containing all ISPs - https://sac.clarityhs.com/report/embed/114996/1
isp_sup_path = r"isp_data\monthly_isp_supplament_2.csv"
isp_sup = pd.read_csv(isp_sup_path)

# Mega dataset: full client + service history (used for last-month service filtering)
mega_path = r"sac_connection_model client_model 2026-04-09T1211.csv"
mega = pd.read_csv(mega_path, low_memory=False)
mega["Service Attendance Date"] = pd.to_datetime(mega["Service Attendance Date"], errors="coerce")
mega["Project Exit Date"] = pd.to_datetime(mega["Project Exit Date"], errors="coerce")

# Filter mega to last month's service events only
mega_last_month = mega[
    (mega["Service Attendance Date"].dt.date >= last_month_start) &
    (mega["Service Attendance Date"].dt.date <= last_month_end)
]
print(f"Service events last month: {len(mega_last_month):,}")


Reporting period: 2026-03-01 → 2026-03-31
Service events last month: 6,630


In [105]:
# Inspect columns to identify date fields for filtering
print("sup columns:", sup.columns.tolist())
print("\nisp_sup columns:", isp_sup.columns.tolist())


sup columns: ['Unnamed: 0', 'Assigned Staff', 'Assigned Staff Home Agency', 'Name', 'Active in Project', 'Project Exit Date', 'Destination', 'Destination Category', 'Unique Identifier', 'Hours Worked Last Week', 'Medicare', 'Days in Project', 'Project Start Date', 'Any Disability', 'Chronic Health', 'Developmental', 'Mental Health', 'Physical', 'Substance Use Disorder', 'Employment Seeking', 'Employment Tenure', 'General Health Status', 'Total Cash Income', 'Total Cash Income.1', 'Last Start Date', 'Count']

isp_sup columns: ['Unnamed: 0', 'Client Full Name', 'Name', 'Updated Date', 'Assigned Staff Home Agency', 'Name.1', 'Added Date', 'Assessment Date', 'Unique Identifier', 'Active in Project', 'Assessing Program']


In [106]:
# Destination counts - last month exits only
# Uses mega dataset filtered to Project Exit Date within the reporting period

dest_exits = mega[
    (mega["Active in Project"] == "No") &
    (mega["Project Exit Date"].dt.date >= last_month_start) &
    (mega["Project Exit Date"].dt.date <= last_month_end)
].drop_duplicates(subset=["Unique Identifier", "Name"])

sup_pivot = pd.pivot_table(
    dest_exits,
    values="Unique Identifier",
    index="Name",
    columns="Destination Category",
    aggfunc="count"
)

dest_count = sup_pivot.style.set_caption(
    f"Destination Counts ({last_month_start.strftime('%B %Y')})"
)


In [107]:
# Total Exits (last month only)

total_exits_raw = mega[
    (mega["Active in Project"] == "No") &
    (mega["Project Exit Date"].dt.date >= last_month_start) &
    (mega["Project Exit Date"].dt.date <= last_month_end)
].drop_duplicates(subset=["Unique Identifier", "Name"])

total_exits_table = pd.pivot_table(
    total_exits_raw, "Unique Identifier", "Name", aggfunc="count"
)
total_exits = total_exits_table.style.set_caption(
    f"Total Exits ({last_month_start.strftime('%B %Y')})"
)


In [108]:
# Total Services Provided (last month only, filtered by Service Attendance Date)

services_pivot = pd.pivot_table(
    mega_last_month,
    values="Count",
    index="Name",
    aggfunc="sum"
)

total_services = services_pivot.style.set_caption(
    f"Total Services Provided ({last_month_start.strftime('%B %Y')})"
)


In [109]:
# Number of unique clients enrolled at any point last month
# A client counts if they started on or before the last day of the month
# AND either exited on or after the first day of the month, or are still active

mega["Project Start Date"] = pd.to_datetime(mega["Project Start Date"], errors="coerce")

enrolled_last_month = mega[
    (mega["Project Start Date"].dt.date <= last_month_end) &
    (
        (mega["Active in Project"] == "Yes") |
        (mega["Project Exit Date"].dt.date >= last_month_start)
    )
].drop_duplicates(subset=["Unique Identifier", "Name"])

clients_served_pivot = pd.pivot_table(
    enrolled_last_month,
    values="Unique Identifier",
    index="Name",
    aggfunc="count"
)

clients_served = clients_served_pivot.style.set_caption(
    f"Unique Clients Enrolled ({last_month_start.strftime('%B %Y')})"
)


In [110]:
# Clients with no services provided last month
# Uses mega dataset: active clients who have no Service Attendance Date in the reporting period

# All currently active clients in the mega dataset
mega_active = mega[mega["Active in Project"] == "Yes"]

# Clients who received at least one service last month
served_last_month_ids = set(mega_last_month["Unique Identifier"].dropna().unique())

# Active clients with no service attendance last month
no_services = mega_active[~mega_active["Unique Identifier"].isin(served_last_month_ids)]

# Deduplicate to one row per client per program
no_services_dedup = no_services.drop_duplicates(subset=["Unique Identifier", "Name"])

no_services_pivot = pd.pivot_table(
    no_services_dedup, "Unique Identifier", "Name", aggfunc="count"
)
no_services_provided = no_services_pivot.style.set_caption(
    f"Clients with no services in {last_month_start.strftime('%B %Y')}"
)


In [ ]:
# Case Manager Ratio
# Uses mega dataset so all enrolled active clients and all assigned staff are captured

active_mega = (
    mega[mega["Active in Project"] == "Yes"]
    .drop_duplicates(subset=["Unique Identifier", "Name"])
)

# Unique case managers per program (from active clients only)
case_managers = (
    active_mega
    .groupby("Name")["Assigned Staff"]
    .nunique()
    .rename("Case Managers")
)

# Unique active clients per program
active_clients = (
    active_mega
    .groupby("Name")["Unique Identifier"]
    .nunique()
    .rename("Active Clients")
)

ratio_df = pd.DataFrame({
    "Active Clients": active_clients,
    "Case Managers": case_managers,
})
ratio_df["Clients per Case Manager"] = (
    ratio_df["Active Clients"] / ratio_df["Case Managers"]
).round(2)

# Keep active_clients as a Series for the ISP goal cell downstream
active_clients_series = active_mega["Unique Identifier"]

cm_ratio = ratio_df.style.set_caption("Case Manager Ratio")


In [112]:
# ISPs Created (last month)

# Concatenate ISP assessment reports
df1 = grove
df2 = rrc
combined_df = pd.concat([df1, df2])

combined_df['Assessment Date'] = pd.to_datetime(combined_df['Assessment Date'], errors='coerce')

last_month_rows = combined_df[
    (combined_df['Assessment Date'].dt.date >= last_month_start) &
    (combined_df['Assessment Date'].dt.date <= last_month_end)
]

created_last_month = pd.pivot_table(last_month_rows, "Unique ID", "Program Name", aggfunc="count")
isps_created_last_month = created_last_month.style.set_caption("ISPs Created")


In [113]:
# Number of ISPs Updated


isp_sup['Updated Date'] = pd.to_datetime(isp_sup['Updated Date'], errors='coerce')

updated_last_month = isp_sup[
    (isp_sup['Updated Date'].dt.date >= last_month_start) &
    (isp_sup['Updated Date'].dt.date <= last_month_end)
]

updated_last_month.columns

updated_last_month_pivot = pd.pivot_table(updated_last_month, "Unique Identifier", "Assessing Program", aggfunc = "nunique")
isps_updated_last_month = updated_last_month_pivot.style.set_caption("Number of ISPs Updated")


In [114]:
# Percentage of Clients with an ISP

# --- Active clients per program (from mega dataset, deduplicated) ---
# Using mega gives us ALL enrolled active clients, not just service-receivers
active_client_pivot = (
    mega[mega["Active in Project"] == "Yes"]
    .drop_duplicates(subset=["Unique Identifier", "Name"])
    .groupby("Name")["Unique Identifier"]
    .nunique()
    .rename("Active Clients")
    .to_frame()
)

# --- Clients with an active ISP per program (unique clients only) ---
isp_summary = isp_sup[["Name.1", "Active in Project", "Unique Identifier"]]
active_isp = isp_summary[isp_summary["Active in Project"] == "Yes"]

active_isp_pivot = (
    active_isp
    .groupby("Name.1")["Unique Identifier"]
    .nunique()
    .rename("Clients With ISP")
    .to_frame()
)
active_isp_pivot.index.name = "Name"

# --- Combine and compute metrics ---
comparison = (
    active_client_pivot
    .join(active_isp_pivot, how="left")
    .fillna(0)
)

comparison[["Active Clients", "Clients With ISP"]] = (
    comparison[["Active Clients", "Clients With ISP"]].astype(int)
)

comparison["Clients Without ISP"] = (
    comparison["Active Clients"] - comparison["Clients With ISP"]
)

comparison["Percent With ISP"] = (
    comparison["Clients With ISP"] / comparison["Active Clients"]
)

pd.set_option("display.expand_frame_repr", False)

percent_of_clients_with_isp = comparison.style.set_caption("Percent of Clients with ISP")


In [115]:
# Percentage of completed ISP Goals

active_isp_responses = combined_df[combined_df["Unique ID"].isin(active_clients["Unique Identifier"])]
active_isp_responses = active_isp_responses.loc[:, ~active_isp_responses.columns.str.startswith('Unnamed')]

df = active_isp_responses

# --- Identify columns ---

start_cols = [c for c in df.columns if c.strip().startswith("Start Date")]
status_cols = [c for c in df.columns if "Status" in c]

# --- Clean start date flags (True if a goal exists for that row & column) ---
start_flags = df[start_cols].replace("", pd.NA).notna()

# --- Status classification masks ---

# Completed = contains "met" or "complete"
completed_mask = df[status_cols].apply(
    lambda col: col.astype(str)
                   .str.lower()
                   .str.contains("met|complete", regex=True, na=False)
)

# In Progress = contains "progress"
inprogress_mask = df[status_cols].apply(
    lambda col: col.astype(str)
                   .str.lower()
                   .str.contains("progress", regex=True, na=False)
)

# --- Per-program counts ---

# Goals started (any non-null start date)
goals_started_by_program = (
    start_flags
    .groupby(df["Program Name"])
    .sum()          # count per start column
    .sum(axis=1)    # sum across all columns
)

# Goals completed
goals_completed_by_program = (
    completed_mask
    .groupby(df["Program Name"])
    .sum()
    .sum(axis=1)
)

# Goals in progress
goals_inprogress_by_program = (
    inprogress_mask
    .groupby(df["Program Name"])
    .sum()
    .sum(axis=1)
)

# --- Assemble final table ---

summary_by_program = pd.DataFrame({
    "Goals Started": goals_started_by_program,
    "Goals Completed": goals_completed_by_program,
    "Goals In Progress": goals_inprogress_by_program,
})

# Fill missing values and ensure integers
summary_by_program = summary_by_program.fillna(0).astype(int)

# Completion rate (%)
summary_by_program["Completion Rate (%)"] = (
    summary_by_program["Goals Completed"] /
    summary_by_program["Goals Started"].replace(0, pd.NA)
) * 100

summary_by_program["Completion Rate (%)"] = (
    summary_by_program["Completion Rate (%)"].fillna(0).round(2)
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)  # don't truncate strings
pd.set_option("display.width", 0)            # auto-detect width
pd.set_option("display.expand_frame_repr", False)


percent_completed_isp = summary_by_program.style.set_caption("Percentage of completed ISP Goals")


In [116]:
# Export all tables to PDF

from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt

def df_to_figure(df, title):
    """Convert a DataFrame to a matplotlib figure with a title."""
    # Handle Styler objects by extracting the underlying DataFrame
    if hasattr(df, 'data'):
        df = df.data

    # Guard against empty DataFrames
    if df.empty:
        fig, ax = plt.subplots(figsize=(8, 2))
        ax.axis('off')
        ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
        ax.text(0.5, 0.5, "No data available", ha='center', va='center',
                transform=ax.transAxes, fontsize=12, color='gray')
        plt.tight_layout()
        return fig

    fig, ax = plt.subplots(figsize=(12, len(df) * 0.5 + 2))
    ax.axis('off')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    
    table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        rowLabels=df.index,
        cellLoc='center',
        loc='center'
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.2, 1.5)
    plt.tight_layout()
    return fig

# Dictionary of all tables with their titles
tables = {
    "Destination Counts": sup_pivot,
    "Total Exits": total_exits_table,
    "Total Services Provided": services_pivot,
    "Clients Served": clients_served_pivot,
    "Clients With No Services": no_services_pivot,
    "Case Manager Ratio": client_case_manager_ratio.to_frame(name="Ratio"),
    "ISPs Created Last Month": created_last_month,
    "ISPs Updated Last Month": updated_last_month_pivot,
    "Percentage of Clients with ISP": comparison,
    "ISP Goal Completion": summary_by_program,
}

# Save to PDF
pdf_path = r"isp_data\analysis_report.pdf"

with PdfPages(pdf_path) as pdf:
    for title, df in tables.items():
        fig = df_to_figure(df, title)
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

print(f"PDF saved to: {pdf_path}")


PDF saved to: isp_data\analysis_report.pdf


In [117]:
dest_count
total_exits
total_services
clients_served 
no_services_provided
cm_ratio
isps_created_last_month
isps_updated_last_month
percent_of_clients_with_isp
percent_completed_isp

,Goals Started,Goals Completed,Goals In Progress,Completion Rate (%)
Program Name,,,,
CITY-FSC: Emergency Bridge Housing at The Grove - ES,162,17,55,10.490000
DHSH-FSC: North A Street Campus - ES,3,0,0,0.000000
DHSH-FSC: Stockton Blvd. Safe Stay - ES,9,3,0,33.330000
